# Data for error analysis

Notebook này dùng để chuẩn bị data mẫu cho công đoạn error analysis. Sử dụng dữ liệu M5 (Walmart sales) và thiết kế các biến giải thích (covariates) cho bài toán dự báo chuỗi thời gian. Dữ liệu M5 bao gồm 3049 sản phẩm bán tại 10 cửa hàng của Walmart trong 5 năm. Dữ liệu có cấu trúc phân cấp theo sản phẩm, cửa hàng, bang, bộ phận, và ngành hàng; ngoài ra còn có các biến giải thích như giá bán, khuyến mãi và lịch các sự kiện.

Output là 1 bảng dạng wide với cột `date`, 7 cột `sales` tương ứng với 7 `dept_id`, và các biến đặc trưng sử dụng chung giữa các series.

In [ ]:
# Tải và giải nén data
import os
import zipfile
import urllib.request

ZIP_URL = "https://github.com/Nixtla/m5-forecasts/raw/main/datasets/m5.zip"
ZIP_FILE = "m5.zip"  
EXTRACTED_FILES = [
    "sales_train_evaluation.csv",
    "calendar.csv",
    "sell_prices.csv"
]

def download_and_extract():
    print("Đang tải zip dữ liệu M5")
    urllib.request.urlretrieve(ZIP_URL, ZIP_FILE)
    print("Tải xong, giải nén")
    with zipfile.ZipFile(ZIP_FILE, 'r') as z:
        for fname in EXTRACTED_FILES:
            z.extract(fname)  
    print("Giải nén xong")

# Kiểm tra 
missing = [f for f in EXTRACTED_FILES if not os.path.exists(f)]
if missing:
    try:
        download_and_extract()
    except Exception as e:
        raise RuntimeError(
            "Không thể tự động tải/giải nén dữ liệu"
            "Tải file m5.zip thủ công, giải nén và đặt các CSV vào thư mục"
            + ZIP_URL
        )
else:
    print("Các file CSV đã tồn tại")

## Load & Prepare

In [1]:
import pandas as pd

# Load data
sales_df = pd.read_csv('sales_train_evaluation.csv')

# Melt to long format
id_cols = ['item_id','dept_id','cat_id','store_id','state_id']
value_cols = [c for c in sales_df.columns if c.startswith('d_')]

sales_long = sales_df.melt(
    id_vars=id_cols, 
    value_vars=value_cols, 
    var_name='d', 
    value_name='sales'
)

# Map d_1 -> date
start_date = pd.to_datetime('2011-01-29')
sales_long['date'] = pd.to_datetime(start_date) + pd.to_timedelta(
    sales_long['d'].str[2:].astype(int) - 1, unit='D'
)

In [2]:
sales_long

,item_id,dept_id,cat_id,store_id,state_id,d,sales,date
0,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29
1,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29
2,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29
3,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29
4,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29
...,...,...,...,...,...,...,...,...
59181085,FOODS_3_823,FOODS_3,FOODS,WI_3,WI,d_1941,1,2016-05-22
59181086,FOODS_3_824,FOODS_3,FOODS,WI_3,WI,d_1941,0,2016-05-22
59181087,FOODS_3_825,FOODS_3,FOODS,WI_3,WI,d_1941,2,2016-05-22
59181088,FOODS_3_826,FOODS_3,FOODS,WI_3,WI,d_1941,0,2016-05-22


In [3]:
# Filter only California
merged = sales_long[sales_long['store_id'] == 'CA_1']

# Giữ 1000 ngày cuối
merged = merged[merged['date'] >= merged['date'].max() - pd.Timedelta(days=999)]
merged.reset_index(drop=True, inplace=True)

In [4]:
merged

,item_id,dept_id,cat_id,store_id,state_id,d,sales,date
0,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_942,0,2013-08-27
1,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_942,0,2013-08-27
2,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_942,0,2013-08-27
3,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_942,0,2013-08-27
4,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_942,3,2013-08-27
...,...,...,...,...,...,...,...,...
3048995,FOODS_3_823,FOODS_3,FOODS,CA_1,CA,d_1941,2,2016-05-22
3048996,FOODS_3_824,FOODS_3,FOODS,CA_1,CA,d_1941,0,2016-05-22
3048997,FOODS_3_825,FOODS_3,FOODS,CA_1,CA,d_1941,1,2016-05-22
3048998,FOODS_3_826,FOODS_3,FOODS,CA_1,CA,d_1941,1,2016-05-22


In [15]:
df_wide = (
    merged[['date', 'item_id', 'sales']]
        .pivot(index='date', columns='item_id', values='sales')
) 

df_wide.columns.name = None

In [16]:
df_wide

,FOODS_1_001,FOODS_1_002,FOODS_1_003,FOODS_1_004,FOODS_1_005,FOODS_1_006,FOODS_1_008,FOODS_1_009,FOODS_1_010,FOODS_1_011,...,HOUSEHOLD_2_507,HOUSEHOLD_2_508,HOUSEHOLD_2_509,HOUSEHOLD_2_510,HOUSEHOLD_2_511,HOUSEHOLD_2_512,HOUSEHOLD_2_513,HOUSEHOLD_2_514,HOUSEHOLD_2_515,HOUSEHOLD_2_516
date,,,,,,,,,,,,,,,,,,,,,
2013-08-27,0,0,1,4,1,3,0,0,0,1,...,0,0,0,1,0,0,1,0,0,0
2013-08-28,0,0,1,8,2,0,0,0,0,0,...,2,0,0,0,0,0,0,0,0,0
2013-08-29,0,0,0,11,3,0,0,4,0,3,...,1,0,0,2,0,0,1,0,0,0
2013-08-30,1,0,1,16,3,4,0,1,0,3,...,1,0,0,0,0,0,0,0,0,0
2013-08-31,0,0,2,10,3,3,0,0,0,0,...,1,1,2,3,0,4,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2016-05-18,0,1,0,2,0,3,0,0,0,0,...,2,0,1,2,2,0,1,0,1,0
2016-05-19,1,0,2,1,0,4,0,1,0,2,...,1,0,0,1,0,2,0,1,0,0
2016-05-20,0,1,2,1,0,0,0,0,2,0,...,1,0,1,0,3,2,1,0,0,1


In [18]:
summary_zero = (
    (df_wide == 0).sum()                 # số zero mỗi cột
        .value_counts()                 # đếm số cột tương ứng
        .sort_index()
        .reset_index()
        .rename(columns={
            'index': 'num_zero_values',
            0: 'num_columns'
        })
)

summary_zero

,num_zero_values,count
0,3,3
1,4,2
2,5,3
3,6,1
4,7,1
...,...,...
855,972,1
856,976,1
857,978,1
858,982,1


In [19]:
df_wide.to_csv('m5_preprocess_data.csv')